# 面试问题：Process Reward Model 与 Outcome Reward Model 有什么区别？怎样训练 step verifier 并引导搜索？

**一句话回答**：Outcome supervision 只评价最终结果，成本低但无法定位第一处错误；Process supervision 为每个中间步骤提供标签，能训练 step-level verifier 并在搜索时早剪枝。PRM 仍可能学到格式捷径，因此要有步骤边界合同、mask、确定性检查、校准和对抗集。

本 Notebook 用 NumPy 手写 step 特征、sigmoid/BCE 梯度训练、变长 mask、路径聚合、active learning、verifier-guided beam、校准拒答和 reward-hacking 反例。


In [ ]:
from dataclasses import dataclass  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

SEED138=13801; rng138=np.random.default_rng(SEED138)  # 计算并保存当前步骤的中间状态。
assert SEED138==13801  # 用受控断言验证关键不变量。
assert math.isclose(1/(1+math.exp(0)),.5)  # 用受控断言验证关键不变量。
assert np.isfinite(rng138.normal())  # 用受控断言验证关键不变量。


## 1. Outcome 标签不能唯一决定过程标签

最终答案正确的路径仍可能包含抵消错误或无效推理；最终错误也不代表第一步就错。process 数据应明确 step segmentation、第一处错误、是否允许跳步及“不确定”标签，并让标注者看到足够上下文但避免未来答案泄漏。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Path138:  # 定义承载本节状态与行为的数据结构。
    steps:tuple; step_labels:tuple; outcome:int  # 执行当前语句以推进本节示例。
    def __post_init__(self):  # 定义本节可复用的核心函数。
        if len(self.steps)!=len(self.step_labels): raise ValueError("step_alignment")  # 按当前条件选择后续控制路径。
p138=Path138(("2+2=5","5-1=4"),(0,0),1)  # 计算并保存当前步骤的中间状态。
assert p138.outcome==1 and p138.step_labels[0]==0  # 用受控断言验证关键不变量。
assert len(p138.steps)==2  # 用受控断言验证关键不变量。
try: Path138(("x",),(1,0),0); raise AssertionError("misaligned")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="step_alignment"  # 捕获预期异常并验证失败分支。


## 2. 教学版 verifier 从可解释 step 特征开始

特征示例包括语法是否合法、局部算术检查、是否引用已有量和是否推进目标。真实 PRM 常用语言模型编码，但数据合同相同。先用可解释 baseline 能发现标签泄漏，例如“因此”或更长文本被误当作正确证据。


In [ ]:
X138=np.array([[1,1,1],[1,1,.8],[1,0,1],[0,0,.5],[1,0,.2],[1,1,.9]],dtype=float)  # 计算并保存当前步骤的中间状态。
y138=np.array([1,1,0,0,0,1],dtype=float)  # 计算并保存当前步骤的中间状态。
assert X138.shape==(6,3)  # 用受控断言验证关键不变量。
assert set(y138)=={0.,1.}  # 用受控断言验证关键不变量。
assert X138[y138==1,1].mean()>X138[y138==0,1].mean()  # 用受控断言验证关键不变量。


## 3. 从零实现 sigmoid、BCE 与梯度下降

对每个 step 输出正确概率，BCE 梯度为 `Xᵀ(p-y)/n`。训练/验证必须按 problem 或题目模板分组切分，不能把同一道题的相邻步骤散到两边。类别不平衡时报告 PR-AUC/recall，而不只看 accuracy。


In [ ]:
def sigmoid138(z):  # 定义本节可复用的核心函数。
    z=np.clip(z,-30,30); return 1/(1+np.exp(-z))  # 计算并保存当前步骤的中间状态。
def train138(X,y,steps=500,lr=.2):  # 定义本节可复用的核心函数。
    w=np.zeros(X.shape[1]); b=0.  # 计算并保存当前步骤的中间状态。
    for _ in range(steps):  # 遍历输入元素以累积或检查结果。
        p=sigmoid138(X@w+b); e=p-y; w-=lr*(X.T@e/len(y)); b-=lr*e.mean()  # 计算并保存当前步骤的中间状态。
    return w,b  # 返回当前分支计算出的结果。
w138,b138=train138(X138,y138); pred138=sigmoid138(X138@w138+b138)  # 计算并保存当前步骤的中间状态。
assert pred138[y138==1].mean()>pred138[y138==0].mean()  # 用受控断言验证关键不变量。
assert np.all((pred138>0)&(pred138<1))  # 用受控断言验证关键不变量。
assert ((pred138>=.5)==y138).mean()>=5/6  # 用受控断言验证关键不变量。


## 4. 变长路径只在真实 step 上计算 loss

batching 后 padding step 必须 mask；若只给“第一处错误”标签，后续步骤可能是不可判定而非负类，也要从 loss 排除。下面展示 masked BCE，改变 padding probability 不影响结果。


In [ ]:
def masked_bce138(probs,labels,mask):  # 定义本节可复用的核心函数。
    p=np.clip(probs,1e-7,1-1e-7); loss=-(labels*np.log(p)+(1-labels)*np.log(1-p)); return float((loss*mask).sum()/mask.sum())  # 计算并保存当前步骤的中间状态。
probs138=np.array([[.9,.8,.01],[.7,.2,.99]]); labels138=np.array([[1,1,0],[1,0,0]]); mask138=np.array([[1,1,0],[1,1,0]])  # 计算并保存当前步骤的中间状态。
base_loss138=masked_bce138(probs138,labels138,mask138); changed138=probs138.copy(); changed138[:,-1]=.5  # 计算并保存当前步骤的中间状态。
assert math.isclose(base_loss138,masked_bce138(changed138,labels138,mask138))  # 用受控断言验证关键不变量。
assert base_loss138>0  # 用受控断言验证关键不变量。
assert mask138.sum()==4  # 用受控断言验证关键不变量。


## 5. 路径分数要惩罚任何关键坏步骤

可用最小 step 概率、概率乘积/对数和或 learned aggregation。平均值会让许多容易正确步骤掩盖一个致命错误；乘积又天然惩罚长路径。选择应与任务失败语义一致，并按长度校准。


In [ ]:
def path_scores138(ps): return {"min":min(ps),"product":float(np.prod(ps)),"mean":float(np.mean(ps))}  # 定义本节可复用的核心函数。
clean138=path_scores138([.9,.9,.9]); one_bad138=path_scores138([.99,.1,.99])  # 计算并保存当前步骤的中间状态。
assert clean138["min"]>one_bad138["min"]  # 用受控断言验证关键不变量。
assert clean138["product"]>one_bad138["product"]  # 用受控断言验证关键不变量。
assert one_bad138["mean"]>.6  # 用受控断言验证关键不变量。


## 6. Active learning 优先标注不确定且有决策价值的步骤

从新 rollout 中选择概率接近 0.5、模型间分歧大、且位于高价值路径分叉处的 step；同时保留随机样本估计无偏总体指标。只标最难样本会改变训练分布，因此要记录 sampling propensity。


In [ ]:
candidates138=[("s1",.95,1.),("s2",.51,2.),("s3",.4,.5),("s4",.7,3.)]  # 计算并保存当前步骤的中间状态。
def acquisition138(p,value): return (1-abs(p-.5)*2)*value  # 定义本节可复用的核心函数。
selected138=max(candidates138,key=lambda x:acquisition138(x[1],x[2]))  # 计算并保存当前步骤的中间状态。
assert selected138[0]=="s2"  # 用受控断言验证关键不变量。
assert acquisition138(.5,2)>acquisition138(.9,2)  # 用受控断言验证关键不变量。
assert all(0<=acquisition138(p,v)<=v for _,p,v in candidates138)  # 用受控断言验证关键不变量。


## 7. Verifier-guided search 在中间层剪枝

generator 为每个状态提出候选 step，PRM 评分后保留 top beam；硬规则失败立即剪掉。PRM 只做启发式，最终仍由 outcome verifier 检查。记录正确分支是否“未生成”还是“评分被剪”，才能决定改 generator 或 verifier。


In [ ]:
tree138={"root":[("a",.9),("b",.4)],"a":[("a1",.8),("a2",.2)],"b":[("b1",.95),("b2",.7)]}  # 计算并保存当前步骤的中间状态。
def beam_step138(frontier,width):  # 定义本节可复用的核心函数。
    cand=[]  # 计算并保存当前步骤的中间状态。
    for node,score in frontier:  # 遍历输入元素以累积或检查结果。
        for child,p in tree138.get(node,[]): cand.append((child,score*p))  # 遍历输入元素以累积或检查结果。
    return sorted(cand,key=lambda x:x[1],reverse=True)[:width]  # 返回当前分支计算出的结果。
level1_138=beam_step138([("root",1.)],2); level2_138=beam_step138(level1_138,2)  # 计算并保存当前步骤的中间状态。
assert level1_138[0][0]=="a"  # 用受控断言验证关键不变量。
assert level2_138[0][0]=="a1"  # 用受控断言验证关键不变量。
assert level2_138[0][1]>.7  # 用受控断言验证关键不变量。


## 8. 校准与对抗集防止格式型 reward hacking

在独立题目上画 reliability、ECE 和错误步骤 recall，按领域/长度/语言切片。加入“格式漂亮但算术错误”“最终正确但过程伪造”“长而重复”等对抗样本；确定性 arithmetic/schema checker 可作为不可被 PRM 高分覆盖的 hard gate。


In [ ]:
def arithmetic_gate138(lhs,rhs): return lhs==rhs  # 定义本节可复用的核心函数。
prm_score138=.97; gate_ok138=arithmetic_gate138(2+2,5)  # 计算并保存当前步骤的中间状态。
decision138=prm_score138>=.8 and gate_ok138  # 计算并保存当前步骤的中间状态。
assert prm_score138>.8  # 用受控断言验证关键不变量。
assert not gate_ok138  # 用受控断言验证关键不变量。
assert not decision138  # 用受控断言验证关键不变量。


## 面试总结

一条完整回答是：**按 problem 切分数据 → 定义 step 边界/第一错/不确定标签 → masked step BCE → 可解释 baseline 查泄漏 → 路径按 min/product 聚合并校准长度 → active learning 记录 propensity → verifier-guided beam 早剪枝 → final outcome verifier 兜底 → 对抗格式捷径与过程伪造 → 分析 generator recall 和 verifier prune error**。PRM 提供更密反馈，但不自动等于真实推理可靠。

延伸阅读：[Let's Verify Step by Step](https://arxiv.org/abs/2305.20050)、[Math-Shepherd](https://arxiv.org/abs/2312.08935)、[DeepSeekMath-V2](https://arxiv.org/abs/2511.22570)。
